In [89]:
#import data
import pandas as pd
df=pd.read_csv("data9.csv")

#cleaning data
df.shape
df=df.dropna(axis=0, how='all')
df=df.dropna(axis=1, how='all')

#data types conversion
df["median wage"] = df["median wage"].str.replace(",", "").astype(float)

df["number of jobs"] = df["number of jobs"].str.replace(",", "").astype(int)

df["mean wage"] = df["mean wage"].str.replace(",", "").astype(float)

for col in range(1, 11):
    df[f"percentile {col}"] = df[f"percentile {col}"].str.replace(",", "")
    df[f"percentile {col}"] = pd.to_numeric(df[f"percentile {col}"], errors="coerce")

df.info()




<class 'pandas.core.frame.DataFrame'>
Index: 45 entries, 0 to 44
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   industry            45 non-null     object 
 1   year                45 non-null     float64
 2   unionisation        45 non-null     float64
 3   number of jobs      45 non-null     int64  
 4   median wage         45 non-null     float64
 5   median wage change  37 non-null     float64
 6   mean wage           45 non-null     float64
 7   mean wage change    36 non-null     float64
 8   percentile 1        44 non-null     float64
 9   percentile 2        44 non-null     float64
 10  percentile 3        44 non-null     float64
 11  percentile 4        45 non-null     int64  
 12  percentile 5        45 non-null     int64  
 13  percentile 6        45 non-null     int64  
 14  percentile 7        45 non-null     int64  
 15  percentile 8        45 non-null     int64  
 16  percentile 9   

/var/folders/gw/297tjlxd4qv8csdzfdw8ndx80000gn/T/ipykernel_28778/606774321.py:3: DtypeWarning: Columns (0,3,4,6,8,9,10,11,12,13,14,15,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("data9.csv")


In [90]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf


results = []

for col in range(1, 11):

   
    data = df[["industry", "year", "unionisation", f"percentile {col}"]].dropna().copy()


    data = data[data[f"percentile {col}"] > 0].copy()

  
    data["log_wage"] = np.log(data[f"percentile {col}"])

    # OLS:
   
    model = smf.ols("log_wage ~ unionisation + C(industry) + C(year)", data=data).fit()

  
    results.append({"percentile": col, "n": len(data), "coefficient": model.params["unionisation"], "p-value": model.pvalues["unionisation"], 
                    "R-squared": model.rsquared})

results_df = pd.DataFrame(results)


print(results_df)

   percentile   n  coefficient   p-value  R-squared
0           1  44     0.008519  0.204540   0.978658
1           2  44     0.006698  0.057433   0.992157
2           3  44     0.004637  0.153358   0.992196
3           4  45     0.003546  0.199074   0.993408
4           5  45     0.004775  0.076617   0.992744
5           6  45     0.004867  0.040315   0.992465
6           7  45     0.003212  0.164321   0.991700
7           8  45     0.003163  0.183000   0.990856
8           9  44     0.003384  0.147253   0.991291
9          10  27     0.001110  0.777217   0.991617
